# SaaS Account-Month OLAP 만들기 초안

이 notebook은 `saas_subscription_churn` raw multi-table 데이터를 바로 feature engineering 하지 않고,
먼저 `account-month` 기준 분석용 OLAP/panel 테이블로 바꾼 뒤 그 위에서 Branch B feature와 target을 만드는 notebook이다.

우리가 지금 하려는 일:

- Branch B에서 보고 싶은 질문은 `membership_lifecycle_risk` 와 `lockin_decay` 를 account-month 단위에서 설명하는 것이다.
- 그러려면 raw multi-table을 그대로 쓰지 말고, account-month grain으로 바꾼 분석용 테이블이 먼저 필요하다.
- 이 notebook은 그 분석용 OLAP 테이블과, 그 위에서 돌아가는 Branch B 학습용 draft를 만든다.

이 notebook의 입력:
- `accounts`
- `subscriptions`
- `feature_usage`
- `churn_events`

이 notebook의 출력:
- `saas_account_month_olap`
  - raw multi-table을 account-month grain으로 정리한 분석용 테이블
- `saas_branch_b_train_draft`
  - 위 OLAP 테이블에서 `b_*` feature와 `target_b_*` target을 만든 학습용 draft

어디서 어떤 컬럼이 만들어지는가:
- `build_subscription_block`
  - current contract, expiry, renewal, next subscription helper
- `build_usage_block`
  - recent/previous/future usage aggregate
- `build_churn_event_block`
  - recent cancel, future churn helper
- `build_branch_b_from_olap`
  - 최종 `b_*` 와 `target_b_*`

핵심 원칙:

- raw table을 한 번에 다 join하지 않는다.
- 기준 grain은 `account-month` 다.
- 각 source table은 그 grain에 맞게 집계한 뒤 붙인다.
- 지금 Branch B에 의미 없는 table은 join하지 않는다.

즉 순서는 `raw multi-table -> account-month OLAP -> Branch B train draft` 이다.

## 진행 순서

- Part 1. 필요한 source table만 로딩
  - Branch B v1에서 실제로 쓸 table만 로딩한다.
- Part 2. account-month base grain 만들기
  - `account_id x month-end` 기준 base row를 만든다.
- Part 3. subscriptions를 account-month block으로 집계
  - current contract, expiry, renewal, plan change 신호를 집계한다.
- Part 4. usage를 account-month block으로 집계
  - recent / previous / future usage를 집계한다.
- Part 5. churn event를 account-month block으로 집계
  - recent cancel과 future churn event helper를 집계한다.
- Part 6. 의미 있는 block만 join해서 OLAP table 만들기
  - block 단위 결과를 하나의 account-month OLAP table로 합친다.
- Part 7. Branch B feature와 target 만들기
  - OLAP table 위에서 `b_*` 와 `target_b_*` 를 만든다.
- Part 8. 최종 export draft 만들기
  - 이후 실험에 바로 쓸 `branch_b_train` draft를 만든다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthEnd

CWD = Path.cwd().resolve()
WORKSPACE_ROOT = next(
    (path for path in [CWD, *CWD.parents] if (path / 'back_research').exists() and (path / 'docs').exists()),
    CWD,
)
NOTEBOOK_ROOT = WORKSPACE_ROOT / 'back_research' / 'wonbeenlee' / 'notebooks' / 'branch_workspaces' / 'membership_subscription_lifecycle'
DATASET_DIR = WORKSPACE_ROOT / 'back_research' / 'wonbeenlee' / 'datasets' / 'saas_subscription_churn'
OUTPUT_DIR = NOTEBOOK_ROOT / 'eda' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLAN_RANK = {'Basic': 0, 'Pro': 1, 'Enterprise': 2}

SAMPLE_BASE_COLUMNS = [
    'sample_id', 'dataset_id', 'entity_id', 'panel_unit', 'as_of_date',
    'join_anchor_date', 'observation_start_date', 'observation_end_date', 'horizon_end_date',
    'horizon_id', 'branch_b_mask', 'mapping_tier', 'sample_weight',
    'y_class_primary', 'y_reg_primary', 'y_churn_power',
    'has_y_class', 'has_y_reg', 'has_y_power',
]

BRANCH_B_FEATURE_COLUMNS = [
    'b_days_since_join', 'b_tenure_days', 'b_subscription_plan_code', 'b_auto_renew_flag',
    'b_days_to_expiry', 'b_cancel_flag_30d', 'b_renewal_failure_count_90d',
    'b_plan_change_up_flag_90d', 'b_plan_change_down_flag_90d',
    'b_usage_intensity_30d', 'b_usage_drop_ratio_30d_vs_prev_30d', 'b_loyalty_benefit_use_30d',
]

BRANCH_B_TARGET_COLUMNS = [
    'target_b_churn_like_event', 'target_b_non_renewal', 'target_b_days_to_failure',
    'target_b_value_loss', 'target_b_lifecycle_risk', 'target_b_lockin_decay',
    'has_target_b_churn_like_event', 'has_target_b_non_renewal', 'has_target_b_days_to_failure',
    'has_target_b_value_loss', 'has_target_b_lifecycle_risk', 'has_target_b_lockin_decay',
]

LEAVE_30D_COLUMNS = [
    'as_of_date',
    'current_plan_tier',
    'current_billing_frequency',
    'current_auto_renew_flag',
    'current_mrr_amount',
    'current_seats',
    'days_since_join',
    'days_to_expiry',
    'cancel_flag_30d',
    'renewal_failure_count_90d',
    'plan_change_up_flag_90d',
    'plan_change_down_flag_90d',
    'usage_intensity_30d',
    'usage_drop_ratio_30d_vs_prev_30d',
    'leave_next_30d',
    'leave_reason',
]

print(DATASET_DIR)
print(DATASET_DIR.exists())

## Part 1. Load Only Relevant Source Tables

이 part에서 하는 일:

- Branch B v1에서 실제로 의미 있는 table만 로딩한다.
- `support_tickets` 는 지금 Branch C 성격이 강하므로 여기서는 join하지 않는다.

이번 OLAP 변환에서 사용할 table:
- `accounts`
- `subscriptions`
- `feature_usage`
- `churn_events`


In [ ]:
def load_saas_tables(dataset_dir: Path) -> dict[str, pd.DataFrame]:
    tables = {
        'accounts': ('ravenstack_accounts.csv', ['signup_date']),
        'subscriptions': ('ravenstack_subscriptions.csv', ['start_date', 'end_date']),
        'feature_usage': ('ravenstack_feature_usage.csv', ['usage_date']),
        'churn_events': ('ravenstack_churn_events.csv', ['churn_date']),
    }
    loaded = {}
    for name, (filename, parse_dates) in tables.items():
        loaded[name] = pd.read_csv(dataset_dir / filename, parse_dates=parse_dates)
    return loaded

saas_tables = load_saas_tables(DATASET_DIR)
pd.DataFrame({
    'table': list(saas_tables.keys()),
    'rows': [frame.shape[0] for frame in saas_tables.values()],
    'cols': [frame.shape[1] for frame in saas_tables.values()],
})

## Part 2. Build Account-Month Base Grain

이 part에서 하는 일:

- root grain을 `account_id x month-end` 로 고정한다.
- 모든 downstream join은 이 base grain에 맞춘 block join이어야 한다.

중요:

- raw multi-table을 직접 join하지 않는다.
- 먼저 base row를 만들고, 각 source table을 그 row에 맞게 집계한다.

In [ ]:
def build_account_month_base(
    accounts: pd.DataFrame,
    subscriptions: pd.DataFrame,
    feature_usage: pd.DataFrame,
    churn_events: pd.DataFrame,
) -> pd.DataFrame:
    max_dates = [
        accounts['signup_date'].max(),
        subscriptions['start_date'].max(),
        subscriptions['end_date'].dropna().max() if subscriptions['end_date'].notna().any() else pd.NaT,
        feature_usage['usage_date'].max(),
        churn_events['churn_date'].max(),
    ]
    calendar_end = max([date for date in max_dates if pd.notna(date)]) + MonthEnd(0)

    rows = []
    for row in accounts[['account_id', 'signup_date']].itertuples(index=False):
        first_month_end = pd.Timestamp(row.signup_date) + MonthEnd(0)
        for as_of_date in pd.date_range(first_month_end, calendar_end, freq='ME'):
            rows.append({
                'dataset_id': 'saas_subscription_churn',
                'entity_id': row.account_id,
                'account_id': row.account_id,
                'panel_unit': 'month',
                'join_anchor_date': pd.Timestamp(row.signup_date),
                'as_of_date': as_of_date,
                'observation_start_date': as_of_date - pd.Timedelta(days=89),
                'observation_end_date': as_of_date,
                'horizon_end_date': as_of_date + pd.Timedelta(days=30),
                'horizon_id': '30d',
                'mapping_tier': 'core',
                'sample_weight': 1.0,
            })

    panel = pd.DataFrame(rows)
    panel['sample_id'] = panel.apply(
        lambda row: f"{row['dataset_id']}:{row['account_id']}:{row['as_of_date'].date()}:{row['horizon_id']}",
        axis=1,
    )
    panel['window_30_start'] = panel['as_of_date'] - pd.Timedelta(days=29)
    panel['window_prev30_start'] = panel['as_of_date'] - pd.Timedelta(days=59)
    panel['window_prev30_end'] = panel['as_of_date'] - pd.Timedelta(days=30)
    panel['future_window_start'] = panel['as_of_date'] + pd.Timedelta(days=1)
    panel['future_window_end'] = panel['horizon_end_date']
    return panel

account_month_base = build_account_month_base(
    saas_tables['accounts'],
    saas_tables['subscriptions'],
    saas_tables['feature_usage'],
    saas_tables['churn_events'],
)
account_month_base.head()

## Part 3. Aggregate Subscriptions Into Account-Month Block

이 part에서 하는 일:

- `subscriptions` raw를 account-month grain으로 집계한다.
- 현재 active contract와 recent contract history를 같은 block으로 정리한다.

이 block에서 만드는 것:
- current plan / seats / value / billing cadence / auto-renew
- days_to_expiry
- recent upgrade / downgrade
- renewal failure helper
- future next subscription helper


In [ ]:
def build_subscription_block(panel: pd.DataFrame, subscriptions: pd.DataFrame) -> pd.DataFrame:
    subs_by_account = {
        account_id: frame.sort_values(['start_date', 'subscription_id']).reset_index(drop=True)
        for account_id, frame in subscriptions.groupby('account_id')
    }

    rows = []
    for row in panel[['sample_id', 'account_id', 'as_of_date', 'observation_start_date', 'horizon_end_date']].itertuples(index=False):
        subs_frame = subs_by_account.get(row.account_id, pd.DataFrame())
        current_sub = None
        next_sub = None
        plan_change_up_flag_90d = 0
        plan_change_down_flag_90d = 0
        renewal_failure_count_90d = 0

        if not subs_frame.empty:
            active_candidates = subs_frame[
                (subs_frame['start_date'] <= row.as_of_date)
                & (subs_frame['end_date'].isna() | (subs_frame['end_date'] >= row.as_of_date))
            ]
            if not active_candidates.empty:
                current_sub = active_candidates.sort_values(['start_date', 'subscription_id']).iloc[-1]

            recent_history = subs_frame[
                (subs_frame['start_date'] >= row.observation_start_date)
                & (subs_frame['start_date'] <= row.as_of_date)
            ]
            if not recent_history.empty:
                plan_change_up_flag_90d = int(recent_history['upgrade_flag'].fillna(False).astype(bool).any())
                plan_change_down_flag_90d = int(recent_history['downgrade_flag'].fillna(False).astype(bool).any())

            for sub in subs_frame.itertuples(index=False):
                if pd.isna(sub.end_date):
                    continue
                if not (row.observation_start_date <= sub.end_date <= row.as_of_date):
                    continue
                next_candidates = subs_frame[subs_frame['start_date'] > sub.end_date].sort_values('start_date')
                renewed_in_grace = not next_candidates.empty and next_candidates.iloc[0]['start_date'] <= sub.end_date + pd.Timedelta(days=30)
                if not renewed_in_grace:
                    renewal_failure_count_90d += 1

            if current_sub is not None and pd.notna(current_sub['end_date']):
                next_candidates = subs_frame[subs_frame['start_date'] > current_sub['end_date']].sort_values('start_date')
                if not next_candidates.empty:
                    next_sub = next_candidates.iloc[0]

        current_plan_tier = current_sub['plan_tier'] if current_sub is not None else pd.NA
        current_auto_renew_flag = current_sub['auto_renew_flag'] if current_sub is not None else pd.NA
        current_end_date = current_sub['end_date'] if current_sub is not None else pd.NaT
        current_seats = current_sub['seats'] if current_sub is not None else np.nan
        current_mrr_amount = current_sub['mrr_amount'] if current_sub is not None else np.nan
        current_arr_amount = current_sub['arr_amount'] if current_sub is not None else np.nan
        current_billing_frequency = current_sub['billing_frequency'] if current_sub is not None else pd.NA
        current_plan_rank = PLAN_RANK.get(current_plan_tier, np.nan) if pd.notna(current_plan_tier) else np.nan
        days_to_expiry = (current_end_date - row.as_of_date).days if pd.notna(current_end_date) else np.nan

        next_subscription_start = next_sub['start_date'] if next_sub is not None else pd.NaT
        next_plan_tier = next_sub['plan_tier'] if next_sub is not None else pd.NA
        next_plan_rank = PLAN_RANK.get(next_plan_tier, np.nan) if pd.notna(next_plan_tier) else np.nan
        next_auto_renew_flag = next_sub['auto_renew_flag'] if next_sub is not None else pd.NA
        next_mrr_amount = next_sub['mrr_amount'] if next_sub is not None else np.nan
        next_seats = next_sub['seats'] if next_sub is not None else np.nan

        due_in_horizon = pd.notna(current_end_date) and (current_end_date <= row.horizon_end_date)
        next_sub_within_grace = next_sub is not None and (next_sub['start_date'] <= current_end_date + pd.Timedelta(days=30)) if pd.notna(current_end_date) else False
        non_renewal_future = int(due_in_horizon and not next_sub_within_grace)
        expiry_without_renewal_future = non_renewal_future
        auto_renew_off_or_fail_future = int(bool(current_auto_renew_flag) and (non_renewal_future == 1 or (next_sub is not None and not bool(next_sub['auto_renew_flag'])))) if pd.notna(current_auto_renew_flag) else 0
        plan_downgrade_future = int(pd.notna(current_plan_rank) and pd.notna(next_plan_rank) and (next_plan_rank < current_plan_rank))
        seat_or_value_contraction_future = int((pd.notna(next_seats) and pd.notna(current_seats) and next_seats < current_seats) or (pd.notna(next_mrr_amount) and pd.notna(current_mrr_amount) and next_mrr_amount < current_mrr_amount))

        rows.append({
            'sample_id': row.sample_id,
            'active_subscription_id': current_sub['subscription_id'] if current_sub is not None else pd.NA,
            'current_plan_tier': current_plan_tier,
            'current_plan_rank': current_plan_rank,
            'current_auto_renew_flag': current_auto_renew_flag,
            'current_end_date': current_end_date,
            'current_seats': current_seats,
            'current_mrr_amount': current_mrr_amount,
            'current_arr_amount': current_arr_amount,
            'current_billing_frequency': current_billing_frequency,
            'current_days_to_expiry': days_to_expiry,
            'plan_change_up_flag_90d': plan_change_up_flag_90d,
            'plan_change_down_flag_90d': plan_change_down_flag_90d,
            'renewal_failure_count_90d': renewal_failure_count_90d,
            'next_subscription_start': next_subscription_start,
            'next_plan_tier': next_plan_tier,
            'next_auto_renew_flag': next_auto_renew_flag,
            'next_mrr_amount': next_mrr_amount,
            'next_seats': next_seats,
            'non_renewal_future': non_renewal_future,
            'expiry_without_renewal_future': expiry_without_renewal_future,
            'auto_renew_off_or_fail_future': auto_renew_off_or_fail_future,
            'plan_downgrade_future': plan_downgrade_future,
            'seat_or_value_contraction_future': seat_or_value_contraction_future,
        })

    return pd.DataFrame(rows)

subscription_block = build_subscription_block(account_month_base, saas_tables['subscriptions'])
subscription_block.head()

## Part 4. Aggregate Usage Into Account-Month Block

이 part에서 하는 일:

- `feature_usage` raw를 account-month grain으로 집계한다.
- raw usage row를 직접 join하지 않고, window aggregate만 만든다.

이 block에서 만드는 것:
- recent 30d usage intensity
- previous 30d usage intensity
- future 30d usage helper
- usage drop ratio helper


In [ ]:
def build_usage_block(panel: pd.DataFrame, subscriptions: pd.DataFrame, feature_usage: pd.DataFrame) -> pd.DataFrame:
    usage = feature_usage.merge(subscriptions[['subscription_id', 'account_id']], on='subscription_id', how='left')
    panel_windows = panel[[
        'sample_id', 'account_id', 'window_30_start', 'as_of_date',
        'window_prev30_start', 'window_prev30_end', 'future_window_start', 'future_window_end',
    ]]
    merged = panel_windows.merge(usage, on='account_id', how='left')

    current_mask = merged['usage_date'].between(merged['window_30_start'], merged['as_of_date'])
    prev_mask = merged['usage_date'].between(merged['window_prev30_start'], merged['window_prev30_end'])
    future_mask = merged['usage_date'].between(merged['future_window_start'], merged['future_window_end'])

    current_usage = merged[current_mask].groupby('sample_id', as_index=False).agg(
        usage_count_30d=('usage_count', 'sum'),
        usage_duration_secs_30d=('usage_duration_secs', 'sum'),
    )
    prev_usage = merged[prev_mask].groupby('sample_id', as_index=False).agg(
        usage_count_prev30=('usage_count', 'sum'),
        usage_duration_secs_prev30=('usage_duration_secs', 'sum'),
    )
    future_usage = merged[future_mask].groupby('sample_id', as_index=False).agg(
        usage_count_future30=('usage_count', 'sum'),
        usage_duration_secs_future30=('usage_duration_secs', 'sum'),
    )

    usage_block = (
        panel[['sample_id']]
        .merge(current_usage, on='sample_id', how='left')
        .merge(prev_usage, on='sample_id', how='left')
        .merge(future_usage, on='sample_id', how='left')
        .fillna(0)
    )

    current_intensity = usage_block['usage_count_30d'] + (usage_block['usage_duration_secs_30d'] / 3600.0)
    prev_intensity = usage_block['usage_count_prev30'] + (usage_block['usage_duration_secs_prev30'] / 3600.0)
    future_intensity = usage_block['usage_count_future30'] + (usage_block['usage_duration_secs_future30'] / 3600.0)

    usage_block['usage_intensity_30d'] = current_intensity
    usage_block['usage_drop_ratio_30d_vs_prev_30d'] = np.where(
        prev_intensity > 0,
        np.clip((prev_intensity - current_intensity) / prev_intensity, 0, 1),
        0.0,
    )
    usage_block['future_usage_intensity_30d'] = future_intensity
    usage_block['future_usage_drop_ratio'] = np.where(
        current_intensity > 0,
        np.clip((current_intensity - future_intensity) / current_intensity, 0, 1),
        0.0,
    )
    return usage_block

usage_block = build_usage_block(account_month_base, saas_tables['subscriptions'], saas_tables['feature_usage'])
usage_block.head()

## Part 5. Aggregate Churn Events Into Account-Month Block

이 part에서 하는 일:

- `churn_events` raw를 account-month grain으로 집계한다.
- 과거 30일 cancel signal과 미래 30일 churn event helper를 만든다.

이 block에서 만드는 것:
- recent cancel flag
- future churn event flag
- days to future churn helper


In [ ]:
def build_churn_event_block(panel: pd.DataFrame, churn_events: pd.DataFrame) -> pd.DataFrame:
    merged = panel[[
        'sample_id', 'account_id', 'window_30_start', 'as_of_date', 'future_window_start', 'horizon_end_date'
    ]].merge(churn_events[['account_id', 'churn_date']], on='account_id', how='left')

    recent = merged[merged['churn_date'].between(merged['window_30_start'], merged['as_of_date'])]
    recent_block = recent.groupby('sample_id', as_index=False).agg(recent_cancel_flag_30d=('churn_date', 'size'))
    recent_block['recent_cancel_flag_30d'] = (recent_block['recent_cancel_flag_30d'] > 0).astype(int)

    future = merged[merged['churn_date'].between(merged['future_window_start'], merged['horizon_end_date'])]
    future_flag = future.groupby('sample_id', as_index=False).agg(future_churn_event_30d=('churn_date', 'size'))
    future_flag['future_churn_event_30d'] = (future_flag['future_churn_event_30d'] > 0).astype(int)

    future_days = future.groupby('sample_id', as_index=False).agg(first_future_churn_date=('churn_date', 'min'))
    future_days = future_days.merge(panel[['sample_id', 'as_of_date']], on='sample_id', how='left')
    future_days['future_churn_days_to_failure'] = (future_days['first_future_churn_date'] - future_days['as_of_date']).dt.days

    churn_block = panel[['sample_id']]
    churn_block = churn_block.merge(recent_block, on='sample_id', how='left')
    churn_block = churn_block.merge(future_flag, on='sample_id', how='left')
    churn_block = churn_block.merge(future_days[['sample_id', 'future_churn_days_to_failure']], on='sample_id', how='left')
    churn_block['recent_cancel_flag_30d'] = churn_block['recent_cancel_flag_30d'].fillna(0).astype(int)
    churn_block['future_churn_event_30d'] = churn_block['future_churn_event_30d'].fillna(0).astype(int)
    return churn_block

churn_event_block = build_churn_event_block(account_month_base, saas_tables['churn_events'])
churn_event_block.head()

## Part 6. Join Only Meaningful Blocks Into OLAP Table

이 part에서 하는 일:

- base grain에 block 단위 집계 결과만 붙인다.
- 의미 없는 raw join은 하지 않는다.

이번 OLAP table에 포함하는 block:
- base account-month
- subscription block
- usage block
- churn event block

이번 단계에서 제외하는 block:
- support_tickets block


In [ ]:
saas_account_month_olap = (
    account_month_base
    .merge(subscription_block, on='sample_id', how='left')
    .merge(usage_block, on='sample_id', how='left')
    .merge(churn_event_block, on='sample_id', how='left')
)
saas_account_month_olap.head()

## Part 7. 30일 이탈 예측용 현재 상태 컬럼과 라벨 만들기

이 part에서 하는 일:

- 이제 raw multi-table이 아니라 `saas_account_month_olap` 위에서만 현재 상태 컬럼과 라벨을 만든다.
- 즉 feature engineering과 label 생성은 OLAP 변환 이후 단계로 둔다.


In [ ]:
def build_leave_next_30d_modeling_base(olap: pd.DataFrame) -> pd.DataFrame:
    frame = olap.copy()

    frame['days_since_join'] = (frame['as_of_date'] - frame['join_anchor_date']).dt.days
    frame['days_to_expiry'] = frame['current_days_to_expiry']
    frame['cancel_flag_30d'] = frame['recent_cancel_flag_30d']
    frame['renewal_failure_count_90d'] = frame['renewal_failure_count_90d']
    frame['plan_change_up_flag_90d'] = frame['plan_change_up_flag_90d']
    frame['plan_change_down_flag_90d'] = frame['plan_change_down_flag_90d']
    frame['usage_intensity_30d'] = frame['usage_intensity_30d']
    frame['usage_drop_ratio_30d_vs_prev_30d'] = frame['usage_drop_ratio_30d_vs_prev_30d']

    frame['leave_next_30d'] = ((frame['future_churn_event_30d'].fillna(0) > 0) | (frame['non_renewal_future'].fillna(0) > 0)).astype(int)
    frame['leave_reason'] = np.select(
        [
            frame['future_churn_event_30d'].fillna(0) > 0,
            frame['non_renewal_future'].fillna(0) > 0,
        ],
        ['churn_event', 'non_renewal'],
        default='stay',
    )
    return frame

saas_leave_next_30d_base = build_leave_next_30d_modeling_base(saas_account_month_olap)
saas_leave_next_30d_base[LEAVE_30D_COLUMNS].head()

## Part 8. 최종 export

이 part에서 하는 일:

- 최종적으로 EDA와 모델링에 쓸 기본 테이블을 만든다.
- source table이 아니라 이미 OLAP 변환된 결과만 내보낸다.
- 결과를 `eda/outputs` 아래로 실제 파일로 저장한다.


In [ ]:
def build_saas_leave_next_30d_modeling_base(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[LEAVE_30D_COLUMNS].sort_values(['as_of_date', 'current_billing_frequency', 'current_plan_tier']).reset_index(drop=True)

saas_leave_next_30d_modeling_base = build_saas_leave_next_30d_modeling_base(saas_leave_next_30d_base)
saas_leave_next_30d_modeling_base.head()

### Export Outputs

이 cell은 아래 두 결과를 `eda/outputs` 아래로 저장한다.

- `saas_account_month_olap`
- `saas_leave_next_30d_modeling_base`

기본적으로 CSV는 항상 저장한다.
Parquet은 현재 환경에 writer dependency가 있으면 같이 저장하고, 없으면 CSV만 남긴다.

In [ ]:
def export_frame(frame: pd.DataFrame, stem: str, output_dir: Path) -> dict[str, str | None]:
    csv_path = output_dir / f'{stem}.csv'
    frame.to_csv(csv_path, index=False)

    parquet_path = output_dir / f'{stem}.parquet'
    parquet_saved = None
    try:
        frame.to_parquet(parquet_path, index=False)
        parquet_saved = str(parquet_path)
    except Exception as exc:
        parquet_saved = f'not saved ({type(exc).__name__}: {exc})'

    return {
        'csv': str(csv_path),
        'parquet': parquet_saved,
    }

export_rows = [
    {'artifact': 'saas_account_month_olap', **export_frame(saas_account_month_olap, 'saas_account_month_olap', OUTPUT_DIR)},
    {'artifact': 'saas_leave_next_30d_modeling_base', **export_frame(saas_leave_next_30d_modeling_base, 'saas_leave_next_30d_modeling_base', OUTPUT_DIR)},
]

pd.DataFrame(export_rows)

## Next Step

이 notebook 다음에 확인할 것:

1. subscription block의 current/next selection 규칙 검증
2. `renewal_failure_count_90d` 와 `leave_next_30d` 정의 샘플 검증
3. `monthly` / `annual` 에 30일 horizon을 같이 쓰는 게 맞는지 검토
4. `saas_account_month_olap` 과 `saas_leave_next_30d_modeling_base` export 확인

지금 구조의 핵심은, 현재 상태를 raw multi-table에서 바로 보지 않고
**account-month OLAP table 위에서 `leave_next_30d` 데이터셋을 만들게 했다** 는 점이다.